In [1]:
import pandas as pd
import gc
import difflib
import unicodedata

Conjunto de funções utilizadas durante a EDA

In [2]:
def normalizar_coluna_municipios(df, nome_coluna):
    """
    Normaliza uma coluna de nomes de municípios em um DataFrame do Pandas:
    - Converte para minúsculas
    - Remove espaços no início e no fim
    - Remove acentos e caracteres especiais
    """
    df[nome_coluna] = (
        df[nome_coluna]
        .astype(str)                       # Garante que seja string
        .str.lower()                       # Converte para minúsculas
        .str.strip()                       # Remove espaços nas pontas
        .str.normalize('NFKD')             # Separa os caracteres dos acentos
        .str.encode('ascii', errors='ignore') # Remove os acentos
        .str.decode('utf-8')               # Retorna para string legível
    )
    return df

# Análise exploratória dos dados de alfabetização do INEP

Os dados de alfabetização baixadados a partir da base do dados do INEP, forma o conjunto principal que irá compor nosso dataset alvo para alimentar o modelo de ML, visto que a variável alvo se encontra neste conjunto de dados, especificamente nos dados dos alunos. Para garantirmos o merge bem sucedido dos diferentes datasets, iremos seguir etapas de qualidade e limpezas dos dados para posteriormente aplicarmos as concatenações e merge de forma eficiente. 

## Ingestão e Verificação dos Dados

Iremos iniciar nossaa análise criando os datasets individuais por área (aluno, município e estado) e ano, para realizarmos as verificações iniciais.

In [3]:
# Dados dos estudantes para cada ano
students2023 = pd.read_csv("alfabetizacao_inep/2023_aluno.csv", sep=";", encoding="latin-1")
students2024 = pd.read_csv("alfabetizacao_inep/2024_aluno.csv", sep=";", encoding="latin-1")
students2025 = pd.read_csv("alfabetizacao_inep/2025_aluno.csv", sep=";", encoding="latin-1")

C:\Users\andre\AppData\Local\Temp\ipykernel_18632\4138985628.py:4: DtypeWarning: Columns (13,14,22,23) have mixed types. Specify dtype option on import or set low_memory=False.
  students2025 = pd.read_csv("alfabetizacao_inep/2025_aluno.csv", sep=";", encoding="latin-1")


In [4]:
# Dados dos municípios por ano
cities2023 = pd.read_csv("alfabetizacao_inep/2023_municipio.csv", sep=";", encoding="latin-1")
cities2024 = pd.read_csv("alfabetizacao_inep/2024_municipio.csv", sep=";", encoding="latin-1")
cities2025 = pd.read_csv("alfabetizacao_inep/2025_municipio.csv", sep=";", encoding="latin-1")

In [5]:
# Dados dos estados por ano
states2023 = pd.read_csv("alfabetizacao_inep/2023_estado.csv", sep=";", encoding="utf-8")
states2024 = pd.read_csv("alfabetizacao_inep/2024_estado.csv", sep=";", encoding="utf-8")
states2025 = pd.read_csv("alfabetizacao_inep/2025_estado.csv", sep=";", encoding="utf-8")

Para entender o DtypeWarning gerado no dataset students2025, iremos verificar primeiro de quais colunas se tratam

In [6]:
# Lista os índices apontados no DtypeWarning
indices_erro = [13, 14, 22, 23]
colunas_com_erro = [students2025.columns[i] for i in indices_erro]
colunas_com_erro

['TX_RESPOSTA_BLOCO_1',
 'TX_GABARITO_BLOCO_1',
 'TX_RESPOSTA_BLOCO_4',
 'TX_GABARITO_BLOCO_4']

Por não entendermos que os dados de resposta e gabarito de questões possam atuar como booas variávies de entrada para o modelo, iremos desconsiderar tais colunas a priori. Entendemos que a informação gerada por tais colunas em conjunto já está inserida nas variáveis de saída, não sendo necessário a manutenção de tais dados no dataset que alimentará o modelo

Agora iremos verificar a paridade entre as variáveis de cada conjunto de datasets

In [7]:
# Dados dos alunos para cada ano abordado
print("Todas as colunas referentes aos alunos em 2023:", students2023.columns.tolist())
print("Todas as colunas referentes aos alunos em 2024:", students2024.columns.tolist())
print("Todas as colunas referentes aos alunos em 2025:", students2025.columns.tolist())

Todas as colunas referentes aos alunos em 2023: ['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'ID_ALUNO', 'TP_SERIE', 'ID_ESCOLA', 'TP_DEPENDENCIA', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'IN_PRESENCA_LP', 'IN_PREENCHIMENTO_LP', 'CO_CADERNO_LP', 'VL_PESO_ALUNO_LP', 'VL_PROFICIENCIA_LP', 'IN_ALFABETIZADO']
Todas as colunas referentes aos alunos em 2024: ['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'ID_ALUNO', 'TP_SERIE', 'ID_ESCOLA', 'TP_DEPENDENCIA', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'IN_PRESENCA_LP', 'IN_PREENCHIMENTO_LP', 'CO_CADERNO_LP', 'VL_PESO_ALUNO_LP', 'VL_PROFICIENCIA_LP', 'IN_ALFABETIZADO']
Todas as colunas referentes aos alunos em 2025: ['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'ID_ALUNO', 'TP_SERIE', 'ID_ESCOLA', 'TP_DEPENDENCIA', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'IN_PRESENCA_LP', 'IN_PREENCHIMENTO_LP', 'CO_CADERNO_LP', 'CO_BLOCO_1', 'TX_RESPOSTA_BLOCO_1', 'TX_GABARITO_BLOCO_1', 'CO_BLOCO_2', 'TX_RESPOSTA_BLOCO_2', 'TX_GABARITO_BLOCO_2', 'CO_BLOCO_3', 'TX_RESPOSTA_BLOCO_3', 'TX_GABARITO_BLOCO_3', 'C

Nos dados referentes aos alunos, notamos que em 2025 há colunas a mais, referentes aos dados de prova e gabarito, já mencionados anteriormente. Esses dados não encontram correspondência com os anos de 2023 e 2024, e não nos parecem relevantes para alimentar o modelo. Por esse motivo, não iremos inclui-los no dataset resultante.

In [8]:
# Dados dos municípios por ano
print("Todas as colunas referentes aos municípios em 2023:", cities2023.columns.tolist())
print("Todas as colunas referentes aos municípios em 2024:", cities2024.columns.tolist())
print("Todas as colunas referentes aos municípios em 2025:", cities2025.columns.tolist())

Todas as colunas referentes aos municípios em 2023: ['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'TP_SERIE', 'ID_TIPO_REDE', 'PC_ALUNO_ALFABETIZADO', 'VL_MEDIA_LP']
Todas as colunas referentes aos municípios em 2024: ['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'TP_SERIE', 'ID_TIPO_REDE', 'PC_ALUNO_ALFABETIZADO', 'VL_MEDIA_LP', 'PC_ALUNO_NIVEL_0_LP', 'PC_ALUNO_NIVEL_1_LP', 'PC_ALUNO_NIVEL_2_LP', 'PC_ALUNO_NIVEL_3_LP', 'PC_ALUNO_NIVEL_4_LP', 'PC_ALUNO_NIVEL_5_LP', 'PC_ALUNO_NIVEL_6_LP', 'PC_ALUNO_NIVEL_7_LP', 'PC_ALUNO_NIVEL_8_LP']
Todas as colunas referentes aos municípios em 2025: ['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'TP_SERIE', 'ID_TIPO_REDE', 'PC_ALUNO_ALFABETIZADO', 'VL_MEDIA_LP', 'PC_ALUNO_NIVEL_0_LP', 'PC_ALUNO_NIVEL_1_LP', 'PC_ALUNO_NIVEL_2_LP', 'PC_ALUNO_NIVEL_3_LP', 'PC_ALUNO_NIVEL_4_LP', 'PC_ALUNO_NIVEL_5_LP', 'PC_ALUNO_NIVEL_6_LP', 'PC_ALUNO_NIVEL_7_LP', 'PC_ALUNO_NIVEL_8_LP']


Encontramos dados referentes às porcentagens dos alunos em cada faixa de aprendizado em relação à classificação do SAEB. Entedemos que esses dados não sejam relevantes para o modelo, visto que não estão presentes em 2023, são explicados por outras variáveis de saída e, por esse motivo, poderiam gerar multicolinearidade no modelo, já que são compostas por estratos das porcentagens dos alunos por faixas de notas e poderiam ser descritas como combinações lineares de outros dados presentes nos datasets.

In [9]:
# Dados dos estados por ano
print("Todas as colunas referentes aos estados em 2023:", states2023.columns.tolist())
print("Todas as colunas referentes aos estados em 2024:", states2024.columns.tolist())
print("Todas as colunas referentes aos estados em 2025:", states2025.columns.tolist())

Todas as colunas referentes aos estados em 2023: ['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'TP_SERIE', 'ID_TIPO_REDE', 'PC_ALUNO_ALFABETIZADO', 'VL_MEDIA_LP']
Todas as colunas referentes aos estados em 2024: ['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'TP_SERIE', 'ID_TIPO_REDE', 'PC_ALUNO_ALFABETIZADO', 'VL_MEDIA_LP', 'PC_ALUNO_NIVEL_0_LP', 'PC_ALUNO_NIVEL_1_LP', 'PC_ALUNO_NIVEL_2_LP', 'PC_ALUNO_NIVEL_3_LP', 'PC_ALUNO_NIVEL_4_LP', 'PC_ALUNO_NIVEL_5_LP', 'PC_ALUNO_NIVEL_6_LP', 'PC_ALUNO_NIVEL_7_LP', 'PC_ALUNO_NIVEL_8_LP']
Todas as colunas referentes aos estados em 2025: ['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'TP_SERIE', 'ID_TIPO_REDE', 'PC_ALUNO_ALFABETIZADO', 'VL_MEDIA_LP', 'PC_ALUNO_NIVEL_0_LP', 'PC_ALUNO_NIVEL_1_LP', 'PC_ALUNO_NIVEL_2_LP', 'PC_ALUNO_NIVEL_3_LP', 'PC_ALUNO_NIVEL_4_LP', 'PC_ALUNO_NIVEL_5_LP', 'PC_ALUNO_NIVEL_6_LP', 'PC_ALUNO_NIVEL_7_LP', 'PC_ALUNO_NIVEL_8_LP']


Pelos mesmos motivos descritos na etapa dos municípios, também não iremos considerar as colunas referentes aos níveis de aprendizado dos alunos que aparecem nos anos de 2024 e 2025, referentes aos dados por estado.

Agora iremos verificar possíveis dados duplicados no id dos alunos. Essa verificação é importante porque esse ID é a variável que irá diferenciar os casos no dataset final. Por esse motivo, garantir a unicidade dos IDs se torna indispensável.

In [10]:
# Dados dos alunos para cada ano abordado
print("Todas as colunas referentes aos alunos em 2023:", students2023['ID_ALUNO'].duplicated().sum())
print("Todas as colunas referentes aos alunos em 2024:", students2024['ID_ALUNO'].duplicated().sum())
print("Todas as colunas referentes aos alunos em 2025:", students2025['ID_ALUNO'].duplicated().sum())

Todas as colunas referentes aos alunos em 2023: 0
Todas as colunas referentes aos alunos em 2024: 0
Todas as colunas referentes aos alunos em 2025: 0


In [11]:
students2023.describe()

,NU_ANO_AVALIACAO,CO_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,CO_CADERNO_LP,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO
count,1747439.0,1.747439e+06,1.747439e+06,1747439.0,1.747439e+06,1.747439e+06,1.747439e+06,1.747439e+06,1.747439e+06,1.747439e+06,1.502809e+06,1.502809e+06,1.747439e+06
mean,2023.0,3.158064e+01,3.164937e+07,2.0,6.002108e+07,2.911219e+00,3.169498e+06,8.601491e-01,8.600066e-01,1.063180e+01,1.160239e+00,7.475638e+02,5.021217e-01
std,0.0,1.080868e+01,1.083407e+07,0.0,1.313293e+04,2.844280e-01,1.082193e+06,3.468324e-01,3.469803e-01,6.185957e+00,4.842851e-01,4.823365e+01,4.999956e-01
min,2023.0,1.100000e+01,1.100000e+07,2.0,6.000000e+07,2.000000e+00,1.100015e+06,0.000000e+00,0.000000e+00,1.000000e+00,9.523810e-02,5.784600e+02,0.000000e+00
25%,2023.0,2.400000e+01,2.400287e+07,2.0,6.000975e+07,3.000000e+00,2.402006e+06,1.000000e+00,1.000000e+00,5.000000e+00,1.000000e+00,7.188371e+02,0.000000e+00
50%,2023.0,3.100000e+01,3.108963e+07,2.0,6.002036e+07,3.000000e+00,3.131307e+06,1.000000e+00,1.000000e+00,1.100000e+01,1.094464e+00,7.521478e+02,1.000000e+00
75%,2023.0,4.100000e+01,4.110591e+07,2.0,6.003424e+07,3.000000e+00,4.119509e+06,1.000000e+00,1.000000e+00,1.600000e+01,1.235216e+00,7.787294e+02,1.000000e+00
max,2023.0,5.200000e+01,5.208382e+07,2.0,6.004281e+07,3.000000e+00,5.222302e+06,1.000000e+00,1.000000e+00,2.100000e+01,1.425454e+02,9.043820e+02,1.000000e+00


In [12]:
students2024.describe()

,NU_ANO_AVALIACAO,CO_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,CO_CADERNO_LP,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO
count,2120560.0,2.120560e+06,2.120560e+06,2120560.0,2.120560e+06,2.120560e+06,2.120560e+06,2.120560e+06,2.120560e+06,2.120560e+06,1.851852e+06,1.851852e+06,2.120560e+06
mean,2024.0,3.249435e+01,3.258714e+07,2.0,6.002267e+07,2.867850e+00,3.265448e+06,8.737258e-01,8.732844e-01,1.074422e+01,1.138954e+00,7.490421e+02,5.220880e-01
std,0.0,1.002957e+01,1.003999e+07,0.0,1.216093e+04,3.386892e-01,1.005170e+06,3.321582e-01,3.326542e-01,6.114784e+00,2.056973e-01,4.704344e+01,4.995120e-01
min,2024.0,1.100000e+01,1.100000e+07,2.0,6.000000e+07,2.000000e+00,1.100015e+06,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,5.938800e+02,0.000000e+00
25%,2024.0,2.600000e+01,2.604746e+07,2.0,6.001258e+07,3.000000e+00,2.610707e+06,1.000000e+00,1.000000e+00,5.000000e+00,1.030000e+00,7.204800e+02,0.000000e+00
50%,2024.0,3.300000e+01,3.307758e+07,2.0,6.002467e+07,3.000000e+00,3.304524e+06,1.000000e+00,1.000000e+00,1.100000e+01,1.090000e+00,7.539500e+02,1.000000e+00
75%,2024.0,4.100000e+01,4.101274e+07,2.0,6.003207e+07,3.000000e+00,4.103503e+06,1.000000e+00,1.000000e+00,1.600000e+01,1.190000e+00,7.803700e+02,1.000000e+00
max,2024.0,5.300000e+01,5.302770e+07,2.0,6.004250e+07,4.000000e+00,5.300108e+06,1.000000e+00,1.000000e+00,4.300000e+01,2.983000e+01,9.008900e+02,1.000000e+00


In [13]:
students2025.describe()

,NU_ANO_AVALIACAO,CO_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,CO_CADERNO_LP,CO_BLOCO_1,CO_BLOCO_2,CO_BLOCO_3,CO_BLOCO_4,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO
count,2222792.0,2.222792e+06,2.222792e+06,2222792.0,2.222164e+06,2.222164e+06,2.222164e+06,2.222792e+06,2.222792e+06,2.222792e+06,1.567060e+06,1.969921e+06,1.969921e+06,402861.000000,1.966605e+06,1.966605e+06,2.222792e+06
mean,2025.0,3.224848e+01,3.238278e+07,2.0,6.002316e+07,2.875741e+00,3.241169e+06,8.862372e-01,8.847454e-01,1.044537e+01,2.582131e+00,7.820045e+00,6.232211e+00,3.999412,1.130269e+00,7.535743e+02,5.862168e-01
std,0.0,1.004266e+01,1.005153e+07,0.0,1.245339e+04,3.298764e-01,1.006450e+06,3.175230e-01,3.193290e-01,6.216472e+00,2.513431e+00,2.820378e+00,2.379316e+00,2.000106,1.429831e-01,4.800590e+01,4.925107e-01
min,2025.0,1.100000e+01,1.100068e+07,2.0,6.000000e+07,2.000000e+00,1.100015e+06,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000,1.000000e+00,5.723559e+02,0.000000e+00
25%,2025.0,2.600000e+01,2.616637e+07,2.0,6.001300e+07,3.000000e+00,2.607901e+06,1.000000e+00,1.000000e+00,5.000000e+00,1.000000e+00,5.000000e+00,5.000000e+00,2.000000,1.041667e+00,7.282354e+02,0.000000e+00
50%,2025.0,3.300000e+01,3.304613e+07,2.0,6.002510e+07,3.000000e+00,3.302502e+06,1.000000e+00,1.000000e+00,1.000000e+01,2.000000e+00,8.000000e+00,6.000000e+00,4.000000,1.095238e+00,7.610449e+02,1.000000e+00
75%,2025.0,3.500000e+01,3.544890e+07,2.0,6.003276e+07,3.000000e+00,3.556206e+06,1.000000e+00,1.000000e+00,1.600000e+01,3.000000e+00,1.000000e+01,8.000000e+00,6.000000,1.181818e+00,7.853025e+02,1.000000e+00
max,2025.0,5.300000e+01,5.306777e+07,2.0,6.004364e+07,3.000000e+00,5.300108e+06,1.000000e+00,1.000000e+00,2.100000e+01,1.300000e+01,1.300000e+01,1.000000e+01,7.000000,2.100000e+01,9.040228e+02,1.000000e+00


A análise dos duplicados para cada dataset, separadamente, mostra que não há duplicação interna. No entanto, ao analisarmos os intervalos numéricos da variável ID_ALUNO em cada ano, via describe(), notamos que grande parte dos IDs se repetem entre os datasets. Como naturalmente a maior parte dos alunos irá se modificar entre os anos por conta da mudança de série dos mesmos, concluímos que o mesmo número de ID foi associado a diferentes alunos de um ano para o outro. Ao verificarmos os valores de ID_ESCOLA nos dados brutos, percebemos que em muitos casos o mesmo ID se refere a escolas em municípios diferentes, de um ano para o outro. Para evitar que essa igualdade dos IDs para alunos e escolas diferentes crie problemas de identificação ao concatenarmos os três datasets, optamos por criar novas máscaras mesclando os IDs com os respectivos anos listados nos csv, antes da concatenação, para garantir a unicidade dos dados.

In [14]:
# Adicionando o ano ao ID para garantir a unicidade do ID do aluno
students2023['ID_ALUNO'] = students2023['ID_ALUNO'].astype('Int64').astype(str).str.strip() + '_2023'
students2024['ID_ALUNO'] = students2024['ID_ALUNO'].astype('Int64').astype(str).str.strip() + '_2024'
students2025['ID_ALUNO'] = students2025['ID_ALUNO'].astype('Int64').astype(str).str.strip() + '_2025'

# O mesmo processo para o ID da escola
students2023['ID_ESCOLA'] = students2023['ID_ESCOLA'].astype('Int64').astype(str).str.strip() + '_2023'
students2024['ID_ESCOLA'] = students2024['ID_ESCOLA'].astype('Int64').astype(str).str.strip() + '_2024'
students2025['ID_ESCOLA'] = students2025['ID_ESCOLA'].astype('Int64').astype(str).str.strip() + '_2025'

## Padronização das variáveis

Agora iremos padronizar os nomes das variáveis dos datasets em inglês e em letras minúsculas. Além disso, iremos selecionar as colunas que serão trabalhadas na etapa de EDA

In [15]:
students_columns = {
    'NU_ANO_AVALIACAO': 'year',
    'CO_UF': 'state_code',
    'ID_ALUNO': 'student_id',
    'TP_SERIE': 'grade',
    'ID_ESCOLA': 'school_id',
    'TP_DEPENDENCIA': 'education_network_type',
    'CO_MUNICIPIO': 'city_code',
    'NO_MUNICIPIO': 'city_name',
    'IN_PRESENCA_LP': 'student_is_present_exam',
    'IN_PREENCHIMENTO_LP': 'student_took_exam',
    'VL_PESO_ALUNO_LP': 'student_exam_weight',
    'VL_PROFICIENCIA_LP': 'student_exam_score',
    'IN_ALFABETIZADO': 'student_is_literate',
}

students2023 = students2023[list(students_columns.keys())].rename(columns=students_columns)
students2024 = students2024[list(students_columns.keys())].rename(columns=students_columns)
students2025 = students2025[list(students_columns.keys())].rename(columns=students_columns)

In [16]:
cities_columns = {
    'NU_ANO_AVALIACAO': 'year',
    'CO_UF': 'state_code',
    'CO_MUNICIPIO': 'city_code',
    'NO_MUNICIPIO': 'city_name', 
    'TP_SERIE': 'grade',
    'ID_TIPO_REDE': 'education_network_type',
    'PC_ALUNO_ALFABETIZADO': 'city_percentage_literate_students',
    'VL_MEDIA_LP': 'city_average_exam_score',
}

cities2023 = cities2023[list(cities_columns.keys())].rename(columns=cities_columns)
cities2024 = cities2024[list(cities_columns.keys())].rename(columns=cities_columns)
cities2025 = cities2025[list(cities_columns.keys())].rename(columns=cities_columns)

In [17]:
states_columns = {
    'NU_ANO_AVALIACAO': 'year',
    'CO_UF': 'state_code',
    'SG_UF': 'state_abbreviation',
    'TP_SERIE': 'grade',
    'ID_TIPO_REDE': 'education_network_type',
    'PC_ALUNO_ALFABETIZADO': 'state_percentage_literate_students',
    'VL_MEDIA_LP': 'state_average_exam_score',
}

states2023 = states2023[list(states_columns.keys())].rename(columns=states_columns)
states2024 = states2024[list(states_columns.keys())].rename(columns=states_columns)
states2025 = states2025[list(states_columns.keys())].rename(columns=states_columns)

## Verificação das Chaves de Junção

Optamos por iniciar nossa abordagem de EDA realizando o merge dos arquivos csv de alunos, municípios e estados dos diferentes anos abordados em um único dataset que será analisado e enriquecido com dados complementares. Para tal, iremos realizar primeiro a concatenação dos datasets por tema (aluno, município e estado) e posterioemente avaliaremos as colunas de junção, para evitar problemas durante o merge.

### Concatenação dos Datasets

In [18]:
# Concatenação dos dados dos alunos
all_students = pd.concat([students2023, students2024, students2025], ignore_index=True)

# Limpeza de memória. Utilizamos essa boa prática para evitar travamentos ao rodar localmente
del students2023, students2024, students2025
gc.collect()

print(f"Total de alunos consolidados: {len(all_students)} registos.")

Total de alunos consolidados: 6090791 registos.


O resultado final é um dataset de alunos com os dados de 2023, 2024 e 2025, com os IDs dos alunos e das escolas com garantia de unicidade e em formato string. Optamos por utilizar a boa prática de apagar os datasets não utilizados e limpar a memória, para reduzir o risco de sobrecarga da memória RAM, visto que estamos trabalhando com datasets com grande número de linhas. O número total de alunos no dataset é importante porque nos guiaremos por esse dado para analisarmos se os merges subsequentes serão bem sucedidos. Agora aplicaremos a mesma etapa aos datasets dos municípios e dos estados.

In [19]:
# Concatenação dos dados dos municípios
all_cities = pd.concat([cities2023, cities2024, cities2025], ignore_index=True)

# Limpeza de memória. Utilizamos essa boa prática para evitar travamentos ao rodar localmente
del cities2023, cities2024, cities2025
gc.collect()

print(f"Total de registros municipais consolidados: {len(all_cities)} registros.")


# Concatenação dos dados dos estados
all_states = pd.concat([states2023, states2024, states2025], ignore_index=True)

# Limpeza de memória.
del states2023, states2024, states2025
gc.collect()

print(f"Total de registros estaduais consolidados: {len(all_states)} registros.")

Total de registros municipais consolidados: 36411 registros.
Total de registros estaduais consolidados: 225 registros.


In [20]:
all_students.isnull().sum()

year                            0
state_code                      0
student_id                      0
grade                           0
school_id                       0
education_network_type        628
city_code                     628
city_name                     628
student_is_present_exam         0
student_took_exam               0
student_exam_weight        769525
student_exam_score         769525
student_is_literate             0
dtype: int64

In [21]:
all_cities.isnull().sum()

year                                 0
state_code                           0
city_code                            0
city_name                            0
grade                                0
education_network_type               0
city_percentage_literate_students    0
city_average_exam_score              0
dtype: int64

In [22]:
all_states.isnull().sum()

year                                  0
state_code                            0
state_abbreviation                    0
grade                                 0
education_network_type                0
state_percentage_literate_students    0
state_average_exam_score              0
dtype: int64

Primeiro iremos buscar as colunas em comum para identificar as colunas candidatas a chaves de junção para o merge.

In [23]:
# Opção 1: Direto com o método .intersection() do Pandas Index
keys_city = all_students.columns.intersection(all_cities.columns).tolist()
keys_state = all_students.columns.intersection(all_states.columns).tolist()

print("Comuns entre alunos e cidades:", keys_city)
print("Comuns entre alunos e estados:", keys_state)

Comuns entre alunos e cidades: ['year', 'state_code', 'grade', 'education_network_type', 'city_code', 'city_name']
Comuns entre alunos e estados: ['year', 'state_code', 'grade', 'education_network_type']


Com a lista das colunas candidatas para o merge, iremos realizar alguns teste para evitar falhas posteriores. Como a lista keys_state está contida na keys_city, sempre que houve necessidade de verificar todas as colunas canditadas a chaves de junção iremos utilizar keys_city para realizar os testes.

### Verificação de nulos

Iniciaremos os testes verificando se há valores nulos em alguma dessas colunas. Essa verificação é importante porque valores nulos podem gerar problemas na etapa de merge, como o aumento no número de linhas.

In [24]:
# Teste de nulos nas chaves de junção
all_students[keys_city].isnull().sum()

year                        0
state_code                  0
grade                       0
education_network_type    628
city_code                 628
city_name                 628
dtype: int64

Observamos 628 dados nulos em três variáveis. Como esse contingente é ínfimo frente ao valor total de registros e a variável city_code não é de fácil imputação, optamos por não utilizar tais dados no dataset do modelo.

In [25]:
# Remoção de nulos focada na coluna city_code
all_students.dropna(subset=['city_code'], inplace=True)

# Teste de nulos nas chaves de junção pós remoção dos nulos de city_code
all_students[keys_city].isnull().sum()

year                      0
state_code                0
grade                     0
education_network_type    0
city_code                 0
city_name                 0
dtype: int64

Após a remoção de nulos da coluna city_code verificamos que não restaram nulos nas outras variáveis, o que dispensa a necessidade de demais tratamentos nesses sentido.

### Checagem de duplicatas

Realizaremos a checagem de duplicatas nos datasets das cidades e estados para evitar que o merge produza aumento do número de linhas no dataset resultante.

In [26]:
# Teste 2: Checagem de duplicatas nas tabelas da direita (Right Tables)
dup_cities = all_cities.duplicated(subset=keys_city).sum()
dup_states = all_states.duplicated(subset=keys_state).sum()

print(f"Duplicatas nas chaves em 'all_cities': {dup_cities}")
print(f"Duplicatas nas chaves em 'all_states': {dup_states}")

Duplicatas nas chaves em 'all_cities': 0
Duplicatas nas chaves em 'all_states': 0


### Comparação de tipagem entre os datasets

In [27]:
df_types = pd.concat([all_students[keys_city].dtypes, all_cities[keys_city].dtypes, all_states[keys_state].dtypes], axis=1)
df_types.columns = ["all_students", "all_cities", "all_states"]
df_types

,all_students,all_cities,all_states
year,int64,int64,int64
state_code,int64,int64,int64
grade,int64,int64,int64
education_network_type,float64,int64,int64
city_code,float64,int64,NaN
city_name,object,object,NaN


Através da comparação entre os tipos para cada dataset, notamos que as variáveis education_network_type e city_code possuem tipos diferentes. Por esse motivo, iremos padronizar os tipos de dados antes do merge. Para reduzir o uso de memória, iremos padronizar também os demais dados das colunas analisadas, com tipos que possam representar seus valores, consumindo menos memória.

In [28]:
# Dicionários com os novos tipos de dados
dtypes_city_keys = {
    'year': 'int16',
    'state_code': 'int8',
    'grade': 'int8',
    'education_network_type': 'int8',
    'city_code': 'int32'
}

dtypes_state_keys = {
    'year': 'int16',
    'state_code': 'int8',
    'grade': 'int8',
    'education_network_type': 'int8'
}

# Pradonização dos tipos
all_students = all_students.astype(dtypes_city_keys)
all_cities = all_cities.astype(dtypes_city_keys)
all_states = all_states.astype(dtypes_state_keys)

In [29]:
df_types = pd.concat([all_students[keys_city].dtypes, all_cities[keys_city].dtypes, all_states[keys_state].dtypes], axis=1)
df_types.columns = ["all_students", "all_cities", "all_states"]
df_types

,all_students,all_cities,all_states
year,int16,int16,int16
state_code,int8,int8,int8
grade,int8,int8,int8
education_network_type,int8,int8,int8
city_code,int32,int32,NaN
city_name,object,object,NaN


Verificando novamente os tipos dos dados, observamos que a padronização foi bem sucedida e que as demais variáveis receberam tipos de dados que sobrecarregam menos a memória.

### Verificação da Correspondência das Chaves

Iremos verificar agora se as combinações entre as chaves de junção do all_students também aparecem no all_cities e no all_states.

In [30]:
# Conjuntos de tuplas com as chaves para comparação
chaves_alunos = set(zip(all_students['year'], all_students['city_code'], all_students['education_network_type']))
chaves_cidades = set(zip(all_cities['year'], all_cities['city_code'], all_cities['education_network_type']))

orfaos = chaves_alunos - chaves_cidades
print(f"Combinações únicas em 'all_students' sem registro em 'all_cities': {len(orfaos)}")

Combinações únicas em 'all_students' sem registro em 'all_cities': 30


In [31]:
# Conjuntos de tuplas com as chaves para comparação (keys_state)
chaves_alunos_estado = set(zip(all_students['year'],all_students['state_code'],all_students['grade'],all_students['education_network_type']))
chaves_estados = set(zip(all_states['year'],all_states['state_code'],all_states['grade'],all_states['education_network_type']))

orfaos_estado = chaves_alunos_estado - chaves_estados
print(f"Combinações únicas em 'all_students' sem registro em 'all_states': {len(orfaos_estado)}")

Combinações únicas em 'all_students' sem registro em 'all_states': 3


Verificamos que de fato há combinações sem correspondência entre os datasets. A quantidade de vezes que cada uma dessas combinações aparecerá durante merge irá determinar o número de dados faltantes. A princípio, iremos realizar o merge mesmo assim e verificar se as variáveis nulas serão numéricas e posteriormente decidiremos o que fazer nesse caso.

In [32]:
# Inspeção rápida das 3 combinações órfãs de estado
df_orfaos_estado = pd.DataFrame(
    list(orfaos_estado), 
    columns=['year', 'state_code', 'grade', 'education_network_type']
)
print(df_orfaos_estado)

   year  state_code  grade  education_network_type
0  2024          53      2                       2
1  2025          17      2                       2
2  2024          17      2                       4


### Validação da Cobertura da Variável education_network_type

Ao analisarmos os dicionários da fonte de dados, observamos que havia classificações da rede escolar que existiam nos datasets de cidades e estados, mas que não apareciam nos dados dos alunos. Para entender melhor esse padrão, iremos avaliar o número de ocorrência de cada tipo de rede, nos três datasets.

In [33]:
# Validação do domínio da variável education_network_type
print("\n1. Redes em 'all_students':")
print(all_students['education_network_type'].value_counts(dropna=False))

print("\n2. Redes em 'all_cities':")
print(all_cities['education_network_type'].value_counts(dropna=False))

print("\n3. Redes em 'all_states':")
print(all_states['education_network_type'].value_counts(dropna=False))


1. Redes em 'all_students':
education_network_type
3    5378617
2     711521
4         25
Name: count, dtype: int64

2. Redes em 'all_cities':
education_network_type
3    16396
5    16019
2     3304
0      692
Name: count, dtype: int64

3. Redes em 'all_states':
education_network_type
5    76
3    75
2    72
0     2
Name: count, dtype: int64


Em primeiro lugar, notamos algo interessante. Há apenas 25 classificações de escola particular (classificação 4) entre os mais de 6 milhões de alunos e nenhuma federal. Por óbvio, esses registros não possuem relevância estatística, alem de não terem correspondentes nos demais datasets. Por esse motivo, tais ocorrências serão deletadas. Outra informação relevantes é que os datasets de cidades e estados possuem redes que não aparecem no all_studentes, em especial a 5, que correspondente à totalização dos dados da rede pública. Por esse motivo, optamos por eliminar todos os registros que não aparecem em all_students, visto que não seria possível utilizar tais dados para treinar o modelo.

In [34]:
# 1. Filtragem para manter estritamente as redes Estadual (2) e Municipal (3)
all_students = all_students[all_students['education_network_type'].isin([2, 3])]
all_cities = all_cities[all_cities['education_network_type'].isin([2, 3])]
all_states = all_states[all_states['education_network_type'].isin([2, 3])]

## Consolidação do dataset de alfabetização

Agora será o merge propriamente dito. Como o merge será feito utilizando city_code, não é necessário utilizar city_name também, visto que na prática carregam a mesma informação. Para evitar duplicação de colunas durante o merge, iremos apagar a coluna city_name do all_students e manter apenas no all_cities.

In [35]:
# Como apagamos dados, iremos verificar novamente o número de linhas de all_students
volumetria_inicial = len(all_students)

# Remoção do city_name das chaves de junção
if 'city_name' in keys_city:
    keys_city.remove('city_name')

# Remoção da coluna city_name de all_students
if 'city_name' in all_students.columns:
    all_students.drop(columns=['city_name'], inplace=True)

# Execução do left join
literate_exam = (
    all_students
    .merge(all_cities, how='left', on=keys_city)
    .merge(all_states, how='left', on=keys_state)
)

#Liberação de memória das tabelas não utilizadas
del all_students, all_cities, all_states
gc.collect()

# Verificação do volume de dados após o merge
volumetria_final = len(literate_exam)

print(f"Linhas em all_students: {volumetria_inicial}")
print(f"Linhas em literate_exam:  {volumetria_final}")


Linhas em all_students: 6090138
Linhas em literate_exam:  6090138


In [36]:
literate_exam.isnull().sum()

year                                       0
state_code                                 0
student_id                                 0
grade                                      0
school_id                                  0
education_network_type                     0
city_code                                  0
student_is_present_exam                    0
student_took_exam                          0
student_exam_weight                   769406
student_exam_score                    769406
student_is_literate                        0
city_name                               4340
city_percentage_literate_students       4340
city_average_exam_score                 4340
state_abbreviation                     27725
state_percentage_literate_students     27725
state_average_exam_score               27725
dtype: int64

## Tratamento dos Nulos

Iniciaremos pelas siglas dos estados. No dataset original dos estados não havia dados faltantes para state_abreviation. 

In [37]:
# Mapeamento de UFs via 'state_code'
uf_map = {
    11: 'RO', 12: 'AC', 13: 'AM', 14: 'RR', 15: 'PA', 16: 'AP', 17: 'TO',
    21: 'MA', 22: 'PI', 23: 'CE', 24: 'RN', 25: 'PB', 26: 'PE', 27: 'AL',
    28: 'SE', 29: 'BA', 31: 'MG', 32: 'ES', 33: 'RJ', 35: 'SP', 41: 'PR',
    42: 'SC', 43: 'RS', 50: 'MS', 51: 'MT', 52: 'GO', 53: 'DF'
}

literate_exam['state_abbreviation'] = (
    literate_exam['state_abbreviation']
    .fillna(literate_exam['state_code'].map(uf_map))
)

print(f"Nulos restantes em 'state_abbreviation': {literate_exam['state_abbreviation'].isnull().sum()}")

Nulos restantes em 'state_abbreviation': 0


Iremos recuperar os nomes das cidades utilizando o próprio dataset literate_exam. Relacionaremos o city_code, que sabemos que não está faltante, com o city_name dos dados que já possuímos e iremos preencher os dados faltantes.

In [38]:
# Bloco 2: Mapeamento interno de nomes de cidades via 'city_code'
city_map = (
    literate_exam.dropna(subset=['city_name'])
    .groupby('city_code')['city_name']
    .first()
)

literate_exam['city_name'] = (
    literate_exam['city_name']
    .fillna(literate_exam['city_code'].map(city_map))
)

print(f"Nulos restantes em 'city_name': {literate_exam['city_name'].isnull().sum()}")

Nulos restantes em 'city_name': 0


Agora iremos utilizar os dados estaduais para realizar a imputação dos dados municipais. Entendemos que por apresentar um dado médio das medições municipais, os valores estaduais servem como bons candidatos para imputação dos dados. Como também há dados estaduais faltando, iremos subir um degrau através do cálculo da média para um dado valor de ano e série e imputar os dados faltante primeiro nos estados e depois nos municípios. Como estamos preenchendo os dados faltantes com dados médios e quantidade total não é da mesma ordem de grandeza do conjunto total de dados, entendemos que não haverá grande impacto do ponto de vista estatístico.

In [39]:
# Bloco 3: Imputação por média de grupo (ano e série) para métricas estaduais
state_lit_avg = literate_exam.groupby(['year', 'grade'])['state_percentage_literate_students'].transform('mean')
state_score_avg = literate_exam.groupby(['year', 'grade'])['state_average_exam_score'].transform('mean')

literate_exam['state_percentage_literate_students'] = (
    literate_exam['state_percentage_literate_students']
    .fillna(state_lit_avg)
)

literate_exam['state_average_exam_score'] = (
    literate_exam['state_average_exam_score']
    .fillna(state_score_avg)
)

print(f"Nulos em 'state_percentage_literate_students': {literate_exam['state_percentage_literate_students'].isnull().sum()}")
print(f"Nulos em 'state_average_exam_score': {literate_exam['state_average_exam_score'].isnull().sum()}")

Nulos em 'state_percentage_literate_students': 0
Nulos em 'state_average_exam_score': 0


In [40]:
# Bloco 3: Fallback hierárquico das métricas municipais usando dados estaduais
literate_exam['city_percentage_literate_students'] = (
    literate_exam['city_percentage_literate_students']
    .fillna(literate_exam['state_percentage_literate_students'])
)

literate_exam['city_average_exam_score'] = (
    literate_exam['city_average_exam_score']
    .fillna(literate_exam['state_average_exam_score'])
)

print(f"Nulos em 'city_percentage_literate_students': {literate_exam['city_percentage_literate_students'].isnull().sum()}")
print(f"Nulos em 'city_average_exam_score': {literate_exam['city_average_exam_score'].isnull().sum()}")

Nulos em 'city_percentage_literate_students': 0
Nulos em 'city_average_exam_score': 0


In [41]:
literate_exam.isnull().sum()

year                                       0
state_code                                 0
student_id                                 0
grade                                      0
school_id                                  0
education_network_type                     0
city_code                                  0
student_is_present_exam                    0
student_took_exam                          0
student_exam_weight                   769406
student_exam_score                    769406
student_is_literate                        0
city_name                                  0
city_percentage_literate_students          0
city_average_exam_score                    0
state_abbreviation                         0
state_percentage_literate_students         0
state_average_exam_score                   0
dtype: int64

Suspeitamos que os dados nulos referentes aos alunos estejam relacionados com os alunos faltante. Iremos desconsiderar esses alunos no treinamento, visto que na nossa visão a ausência desses dados reflete a realidade e a imputação de dados poderia criar uma distorção. Para termos certeza, iremos verificar quantos dados nulos se referem a alunos faltantes.

In [42]:
# Número de pessoas que faltaram a prova
faltantes = (literate_exam['student_is_present_exam'] == 0).sum()
print(f"Total de alunos que faltaram: {faltantes}")

# Número de notas nulas filtrando apenas pelos faltantes
nulos_faltantes = literate_exam[literate_exam['student_is_present_exam'] == 0]['student_exam_score'].isnull().sum()
print(f"Total de notas nulas entre os faltantes: {nulos_faltantes}")

Total de alunos que faltaram: 764905
Total de notas nulas entre os faltantes: 764905


Comparando a quantidade de dados nulos dos alunos com o número de alunos faltantes, notamos que há apenas um aluno que teria comparecido, mas que está com os dados nulos. Optamos por apenas alterar o registro dele para faltante para que quando selecionarmos apenas os alunos não faltantes para o treinamento do modelo ele seja retirado junto, visto que é apenas um e que sequer será usado para o treinamento.

In [43]:
# Reclassifica a presença do aluno anômalo para 0 (ausente/inválido)
literate_exam.loc[
    (literate_exam['student_exam_score'].isnull()) & 
    (literate_exam['student_is_present_exam'] == 1), 
    'student_is_present_exam'
] = 0

Esse foi o resultado final do tratamento realizado sobre o dataset literate_exam

In [44]:
literate_exam.head(10)

,year,state_code,student_id,grade,school_id,education_network_type,city_code,student_is_present_exam,student_took_exam,student_exam_weight,student_exam_score,student_is_literate,city_name,city_percentage_literate_students,city_average_exam_score,state_abbreviation,state_percentage_literate_students,state_average_exam_score
0,2023,11,11008701_2023,2,60000001_2023,3,1100205,0,0,NaN,NaN,0,Porto Velho,65.06,761.5118,RO,65.17,760.1971
1,2023,11,11008695_2023,2,60000001_2023,3,1100205,1,1,1.045465,714.314857,0,Porto Velho,65.06,761.5118,RO,65.17,760.1971
2,2023,11,11008687_2023,2,60000001_2023,3,1100205,0,0,NaN,NaN,0,Porto Velho,65.06,761.5118,RO,65.17,760.1971
3,2023,11,11008682_2023,2,60000001_2023,3,1100205,1,1,1.045465,759.206313,1,Porto Velho,65.06,761.5118,RO,65.17,760.1971
4,2023,11,11008729_2023,2,60000001_2023,3,1100205,0,0,NaN,NaN,0,Porto Velho,65.06,761.5118,RO,65.17,760.1971
5,2023,11,11008665_2023,2,60000001_2023,3,1100205,1,1,1.632346,796.026914,1,Porto Velho,65.06,761.5118,RO,65.17,760.1971
6,2023,11,11008730_2023,2,60000001_2023,3,1100205,1,1,1.845623,754.646186,1,Porto Velho,65.06,761.5118,RO,65.17,760.1971
7,2023,11,11008726_2023,2,60000001_2023,3,1100205,0,0,NaN,NaN,0,Porto Velho,65.06,761.5118,RO,65.17,760.1971
8,2023,11,11008667_2023,2,60000001_2023,3,1100205,1,1,1.632346,796.026914,1,Porto Velho,65.06,761.5118,RO,65.17,760.1971
9,2023,11,11008680_2023,2,60000001_2023,3,1100205,1,1,1.632346,796.026914,1,Porto Velho,65.06,761.5118,RO,65.17,760.1971


In [45]:
literate_exam.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6090138 entries, 0 to 6090137
Data columns (total 18 columns):
 #   Column                              Dtype  
---  ------                              -----  
 0   year                                int16  
 1   state_code                          int8   
 2   student_id                          object 
 3   grade                               int8   
 4   school_id                           object 
 5   education_network_type              int8   
 6   city_code                           int32  
 7   student_is_present_exam             int64  
 8   student_took_exam                   int64  
 9   student_exam_weight                 float64
 10  student_exam_score                  float64
 11  student_is_literate                 int64  
 12  city_name                           object 
 13  city_percentage_literate_students   float64
 14  city_average_exam_score             float64
 15  state_abbreviation                  object 
 16  

## Enriquecimento do Dataset

Primeiro iremos carregar todos os datasets para analisá-los antes do merge. Como a quantidade de chaves utilizadas é pequena, e os dados já passaram por um pré-processamento, iremos fazer verificações rápidas para garantir que não haja erros durante o merge.

In [46]:
# Carrega a base do FUNDEB
fundeb = pd.read_csv("fundeb/fundeb.csv", sep=";", encoding="utf-8")
print("--- ESTRUTURA DA BASE FUNDEB ---")
print(fundeb.info())

--- ESTRUTURA DA BASE FUNDEB ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16710 entries, 0 to 16709
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   year                16710 non-null  int64  
 1   city_code           16710 non-null  int64  
 2   city_name           16710 non-null  object 
 3   state_abbreviation  16710 non-null  object 
 4   fundeb_amount       16710 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 652.9+ KB
None


In [47]:
fundeb.isnull().sum()

year                  0
city_code             0
city_name             0
state_abbreviation    0
fundeb_amount         0
dtype: int64

In [48]:
# Carrega a base do IDHM
idhm = pd.read_csv("idhm/idhm_uf_2022_2024.csv", sep=";", encoding="utf-8")
print("\n--- ESTRUTURA DA BASE IDHM ---")
print(idhm.info())


--- ESTRUTURA DA BASE IDHM ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 36 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   year                                 81 non-null     int64  
 1   state_code                           81 non-null     int64  
 2   state_name                           81 non-null     object 
 3   hdi                                  81 non-null     float64
 4   hdi_life_expectancy                  81 non-null     float64
 5   hdi_education                        81 non-null     float64
 6   hdi_income                           81 non-null     float64
 7   education_subindex                   81 non-null     float64
 8   school_attendance_subindex           81 non-null     float64
 9   school_attendance_rate_5_6           81 non-null     float64
 10  elementary_completion_rate_11_13     81 non-null     float64
 11  el

In [49]:
idhm.isnull().sum()

year                                   0
state_code                             0
state_name                             0
hdi                                    0
hdi_life_expectancy                    0
hdi_education                          0
hdi_income                             0
education_subindex                     0
school_attendance_subindex             0
school_attendance_rate_5_6             0
elementary_completion_rate_11_13       0
elementary_completion_rate_15_17       0
high_school_completion_rate_18_20      0
elementary_completion_rate_18_plus     0
expected_years_of_schooling            0
illiteracy_rate_15_plus                0
illiteracy_rate_18_plus                0
illiteracy_rate_25_plus                0
school_attendance_rate_6_14            0
school_attendance_rate_15_17           0
school_attendance_rate_18_24           0
elementary_completion_rate_18_24       0
elementary_completion_rate_25_plus     0
high_school_completion_rate_25_plus    0
age_lag_2y_basic

In [50]:
# Carregar a base de dados do Censo 2022
censo_2022 = pd.read_csv("censo_2022/censo_2022.csv",sep=";", encoding="utf-8")
print("\n--- ESTRUTURA DA BASE DADOS DO CENSO 2022 ---")
print(censo_2022.info())


--- ESTRUTURA DA BASE DADOS DO CENSO 2022 ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   city_code              5570 non-null   int64  
 1   state_abbreviation     5570 non-null   object 
 2   households             5570 non-null   int64  
 3   area                   5570 non-null   int64  
 4   literacy_rate          5570 non-null   float64
 5   median_age             5570 non-null   float64
 6   sex_ratio              5570 non-null   float64
 7   aging_index            5570 non-null   float64
 8   city_name              5570 non-null   object 
 9   population_total       5570 non-null   int64  
 10  population_white       5570 non-null   int64  
 11  population_black       5570 non-null   int64  
 12  population_yellow      5570 non-null   int64  
 13  population_brown       5570 non-null   int64  
 14  populatio

In [51]:
censo_2022.isnull().sum()

city_code                0
state_abbreviation       0
households               0
area                     0
literacy_rate            0
median_age               0
sex_ratio                0
aging_index              0
city_name                0
population_total         0
population_white         0
population_black         0
population_yellow        0
population_brown         0
population_indigenous    0
dtype: int64

In [52]:
# Carrega a base de Dados do Censo Escolar 2023, 2024 e 2025
censo_escolar2023 = pd.read_csv("saida/censo_escolar_municipal_2023.csv",sep=",", encoding="utf-8")
censo_escolar2024 = pd.read_csv("saida/censo_escolar_municipal_2024.csv",sep=",", encoding="utf-8")
censo_escolar2025 = pd.read_csv("saida/censo_escolar_municipal_2025.csv",sep=",", encoding="utf-8")

# Concatenação das bases de dados do censo escolar
censo_escolar = pd.concat([censo_escolar2023, censo_escolar2024, censo_escolar2025], ignore_index=True)

del censo_escolar2023, censo_escolar2024, censo_escolar2025
gc.collect()

print("\n--- ESTRUTURA DA BASE DADOS ESCOLARES (CENSO ESCOLAR) ---")
print(censo_escolar.info())


--- ESTRUTURA DA BASE DADOS ESCOLARES (CENSO ESCOLAR) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16711 entries, 0 to 16710
Data columns (total 16 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   ano                                 16711 non-null  int64  
 1   id_municipio                        16711 non-null  int64  
 2   qtd_escolas                         16711 non-null  int64  
 3   total_matriculas                    16711 non-null  float64
 4   total_docentes                      16711 non-null  float64
 5   total_turmas                        16711 non-null  float64
 6   pct_escolas_rurais                  16711 non-null  float64
 7   pct_escolas_internet                16711 non-null  float64
 8   pct_escolas_biblioteca              16711 non-null  float64
 9   pct_escolas_sala_leitura            16711 non-null  float64
 10  pct_escolas_espaco_leitura          16711 non-n

In [53]:
censo_escolar.isnull().sum()

ano                                   0
id_municipio                          0
qtd_escolas                           0
total_matriculas                      0
total_docentes                        0
total_turmas                          0
pct_escolas_rurais                    0
pct_escolas_internet                  0
pct_escolas_biblioteca                0
pct_escolas_sala_leitura              0
pct_escolas_espaco_leitura            0
pct_matriculas_integral               0
alunos_por_docente                    0
alunos_por_turma                      0
pct_matriculas_escolas_turmas_zero    0
qtd_escolas_docentes_nulos            0
dtype: int64

Como os dados do censo_escolar estão com as variáveis em português, iremos padronizar os nomes das colunas em inglês, como estabelecido nos demais datasets.

In [54]:
# Renomeia as variáveis (Padronização completa para o Inglês)
renomeacao_censo = {
    "ano": "year",
    "id_municipio": "city_code",
    "qtd_escolas": "school_count",
    "total_matriculas": "total_enrollments",
    "total_docentes": "total_teachers",
    "total_turmas": "total_classes",
    "pct_escolas_rurais": "pct_rural_schools",
    "pct_escolas_internet": "pct_internet_schools",
    "pct_escolas_biblioteca": "pct_library_schools",
    "pct_escolas_sala_leitura": "pct_reading_room_schools",
    "pct_escolas_espaco_leitura": "pct_reading_space_schools",
    "pct_matriculas_integral": "pct_full_time_enrollments",
    "alunos_por_docente": "students_per_teacher",
    "alunos_por_turma": "students_per_class",
    "pct_matriculas_escolas_turmas_zero": "pct_enrollments_zero_class_schools",
    "qtd_escolas_docentes_nulos": "null_teacher_schools_count"
}

censo_escolar = censo_escolar.rename(columns=renomeacao_censo)

# Validação da alteração
print(f"Colunas atuais do Censo Escolar: {censo_escolar.columns.tolist()}")

Colunas atuais do Censo Escolar: ['year', 'city_code', 'school_count', 'total_enrollments', 'total_teachers', 'total_classes', 'pct_rural_schools', 'pct_internet_schools', 'pct_library_schools', 'pct_reading_room_schools', 'pct_reading_space_schools', 'pct_full_time_enrollments', 'students_per_teacher', 'students_per_class', 'pct_enrollments_zero_class_schools', 'null_teacher_schools_count']


Com os nomes das variáveis padronizados, iremos para a etapa de padronização dos tipos. Nós já haviamos alterado alguns tipos do literacy_exam para reduzir o uso de memória. Por esse motivo, iremos compatibilizar os tipos dos demais datasets, com os já alterados do dataset principal.

In [55]:
# Padronização de Tipos

# city_code para int32 nas bases municipais
bases_com_city = [literate_exam, fundeb, censo_2022, censo_escolar]

for df in bases_com_city:
    if 'city_code' in df.columns:
        df['city_code'] = df['city_code'].astype('int32')

# state_code para int8 nas bases que o possuem
literate_exam['state_code'] = literate_exam['state_code'].astype('int8')
idhm['state_code'] = idhm['state_code'].astype('int8')

# Verificação das alterações
print("Tipagem atualizada com sucesso:")
print(f"literacy_exam (city_code, state_code): {literate_exam['city_code'].dtype}, {literate_exam['state_code'].dtype}")
print(f"censo_escolar (city_code): {censo_escolar['city_code'].dtype}")
print(f"fundeb (city_code): {fundeb['city_code'].dtype}")
print(f"censo_2022 (city_code): {censo_2022['city_code'].dtype}")
print(f"idhm (state_code): {idhm['state_code'].dtype}")

Tipagem atualizada com sucesso:
literacy_exam (city_code, state_code): int32, int8
censo_escolar (city_code): int32
fundeb (city_code): int32
censo_2022 (city_code): int32
idhm (state_code): int8


Agora iremos validar a unicidade das chaves. Esse passo garante que não existem chaves duplicadas nas tabelas dimensionais, o que previne a multiplicação indevida de linhas na hora do merge.

In [56]:
# Validação de Unicidade
print("Verificação de Duplicidades:")
print(f"Censo 2022 (city_code único?): {censo_2022['city_code'].is_unique}")
print(f"Censo Escolar (city_code + year único?): {censo_escolar.duplicated(subset=['city_code', 'year']).sum() == 0}")
print(f"FUNDEB (city_name + year único?): {fundeb.duplicated(subset=['city_name', 'year']).sum() == 0}")
print(f"IDHM (state_code + year único?): {idhm.duplicated(subset=['state_code', 'year']).sum() == 0}")

Verificação de Duplicidades:
Censo 2022 (city_code único?): True
Censo Escolar (city_code + year único?): True
FUNDEB (city_name + year único?): False
IDHM (state_code + year único?): True


Durante uma anáise visual dos dados, quando baixamos os arquivos brutos, verificamos que o city_code do FUNDEB é uma máscara. Por esse motivo, teremos que utilizar o city_name no lugar e apagar a coluna city_code do FUNDEB para evitar duplicações. Iremos realizar previamente uma padronização e comparação dos nomes dos municípios no fundeb e no literate_exam, para encontrar possíveis erros.

In [57]:
# Normalização de Strings
def limpar_texto(texto):
    if pd.isna(texto):
        return texto
    # Converte para string, remove acentos, passa para minúsculo e tira espaços extras nas pontas
    texto_str = str(texto)
    texto_sem_acento = unicodedata.normalize('NFKD', texto_str).encode('ASCII', 'ignore').decode('utf-8')
    return texto_sem_acento.lower().strip()

print("Normalizando nomes das cidades em literate_exam e fundeb...")
literate_exam['city_name_norm'] = literate_exam['city_name'].apply(limpar_texto)
fundeb['city_name_norm'] = fundeb['city_name'].apply(limpar_texto)

# Validação de Intersecção 
municipios_exam = set(literate_exam['city_name_norm'].dropna())
municipios_fundeb = set(fundeb['city_name_norm'].dropna())

match_perfeito = municipios_fundeb.intersection(municipios_exam)
sem_match = municipios_fundeb - municipios_exam

print(f"\n--- DIAGNÓSTICO DO MERGE POR TEXTO ---")
print(f"Nomes únicos no literate_exam: {len(municipios_exam)}")
print(f"Nomes únicos no fundeb: {len(municipios_fundeb)}")
print(f"Municípios com match perfeito: {len(match_perfeito)}")
print(f"Municípios no fundeb SEM match na base principal: {len(sem_match)}")

if len(sem_match) > 0:
    print("\nExemplos de municípios sem correspondência:")
    print(list(sem_match)[:10])

# Exclusão do Código Mascarado
fundeb = fundeb.drop(columns=['city_code'], errors='ignore')

Normalizando nomes das cidades em literate_exam e fundeb...

--- DIAGNÓSTICO DO MERGE POR TEXTO ---
Nomes únicos no literate_exam: 5287
Nomes únicos no fundeb: 5290
Municípios com match perfeito: 5254
Municípios no fundeb SEM match na base principal: 36

Exemplos de municípios sem correspondência:
['moji mirim', 'fortaleza do tabocao', 'santana do livramento', 'sao domingos de pombal', 'poxoreo', 'lagoa do itaenga', 'gracho cardoso', 'trajano de morais', 'belem de sao francisco', 'eldorado dos carajas']


Encontramos uma quantidade relevante de municípios sem match entre os dois datasets. Utilizaremos a biblioteca difflib para tentar compartibiizar os nomes. Caso não consigamos, será necessário abrir mão do dataset fundeb, visto que não haverá como realizar o merge.

In [58]:
print("Iniciando correção de nomes corrompidos por similaridade (dentro do mesmo Estado)...")

mapeamento_cidades = {}
estados = fundeb['state_abbreviation'].dropna().unique()

for uf in estados:
    # Filtra as cidades válidas da base principal para aquele Estado
    cidades_validas = literate_exam[literate_exam['state_abbreviation'] == uf]['city_name_norm'].dropna().unique()
    
    # Filtra as cidades do fundeb para o mesmo Estado
    cidades_fundeb = fundeb[fundeb['state_abbreviation'] == uf]['city_name_norm'].dropna().unique()
    
    # Procura os que não deram match perfeito
    for cidade_f in cidades_fundeb:
        if cidade_f not in cidades_validas:
            # Encontra a string mais parecida (cutoff=0.6 garante uma boa margem)
            match = difflib.get_close_matches(cidade_f, cidades_validas, n=1, cutoff=0.8)
            if match:
                mapeamento_cidades[cidade_f] = match[0]

# Substitui os nomes corrompidos no fundeb pelas strings validadas
fundeb['city_name_norm'] = fundeb['city_name_norm'].replace(mapeamento_cidades)

# Nova Auditoria
municipios_exam_novo = set(literate_exam['city_name_norm'].dropna())
municipios_fundeb_novo = set(fundeb['city_name_norm'].dropna())

match_perfeito_novo = municipios_fundeb_novo.intersection(municipios_exam_novo)
sem_match_novo = municipios_fundeb_novo - municipios_exam_novo

print(f"\n--- RESULTADO DA CORREÇÃO ---")
print(f"Municípios corrigidos automaticamente: {len(mapeamento_cidades)}")
print(f"Novo total com match perfeito: {len(match_perfeito_novo)}")
print(f"Municípios ainda sem match: {len(sem_match_novo)}")

if len(sem_match_novo) > 0:
    print(f"\nRestantes ({len(sem_match_novo)}):")
    print(list(sem_match_novo)[:15])

Iniciando correção de nomes corrompidos por similaridade (dentro do mesmo Estado)...

--- RESULTADO DA CORREÇÃO ---
Municípios corrigidos automaticamente: 27
Novo total com match perfeito: 5279
Municípios ainda sem match: 10

Restantes (10):
['fortaleza do tabocao', 'sao domingos de pombal', 'embu', 'sao valerio da natividade', 'rolador', 'serido', 'barra do rio azul', 'coqueiro baixo', 'augusto severo', 'uniao da serra']


A abordagem adotada foi bem sucedida, por esse motivo manteremos o fundeb como dataset para o merge.

In [59]:
# Verifica quantas duplicatas foram geradas pelo mapeamento do difflib
duplicatas_fundeb = fundeb.duplicated(subset=['city_name_norm', 'state_abbreviation', 'year']).sum()
print(f"Duplicatas encontradas no FUNDEB após correções de texto: {duplicatas_fundeb}")

# Exibe exemplos das duplicatas (opcional, para visualização)
if duplicatas_fundeb > 0:
    print("\nExemplos de linhas duplicadas:")
    display(fundeb[fundeb.duplicated(subset=['city_name_norm', 'state_abbreviation', 'year'], keep=False)].head())

# Remove as duplicatas mantendo o primeiro registro
fundeb = fundeb.drop_duplicates(subset=['city_name_norm', 'state_abbreviation', 'year'])

# Valida a unicidade da chave composta
chaves_unicas = fundeb.duplicated(subset=['city_name_norm', 'state_abbreviation', 'year']).sum() == 0
print(f"\nChave (city_name_norm, state_abbreviation, year) é única no FUNDEB? {chaves_unicas}")

Duplicatas encontradas no FUNDEB após correções de texto: 0

Chave (city_name_norm, state_abbreviation, year) é única no FUNDEB? True


Não encontramos dados atualizados de 2025 para o idhm. Por esse motivo utilizaremos uma estratégia de lag. Iremos utilizar o fundeb como "dados do ano anterior". Ou seja, os dados de 2023 do dataset principal serão preenchidos com dados do idhm de 2022, para evitar dados nulos em um ano inteiro. Além disso, utilizaremos a raça em termos de porcentagem. Optamos por essa abordagem imaginando que a normalização desses dados não iria sobrecarregar tanto o modelo, mas mantivemos a população total, para ter um absoluto que leve em conta a diferença populacional dos municípios.

In [60]:
# Deslocamento Temporal (Lag) para o IDHM
idhm['year_join'] = idhm['year'] + 1

# Proporcionalidade Demográfica (Censo 2022)
racas = ['white', 'black', 'yellow', 'brown', 'indigenous']
for raca in racas:
    coluna_abs = f'population_{raca}'
    coluna_pct = f'pct_{raca}'
    censo_2022[coluna_pct] = censo_2022[coluna_abs] / censo_2022['population_total']

# Remove as colunas de população absoluta, mantendo apenas a population_total
censo_2022 = censo_2022.drop(columns=[f'population_{raca}' for raca in racas])

# Validação das alterações
print(f"Colunas atuais do Censo 2022: {censo_2022.columns.tolist()}")
print(f"'year_join' presente no IDHM: {'year_join' in idhm.columns}")

Colunas atuais do Censo 2022: ['city_code', 'state_abbreviation', 'households', 'area', 'literacy_rate', 'median_age', 'sex_ratio', 'aging_index', 'city_name', 'population_total', 'pct_white', 'pct_black', 'pct_yellow', 'pct_brown', 'pct_indigenous']
'year_join' presente no IDHM: True


Segundo o Ipea, os dados a nível municipal ainda do IDHM não estão disponíveis, por conta de atraso no tratamento dos dados do censo de 2022. Por esse motivo, o Ipea fornece apenas os dados a nível estadual, considerando os dados posteiores a 2020. Iremos manter as variáveis agregadas para evitar a ocorrência de multicolinearidade, visto que são muitas colunas com dados que representam informações próximas entre si.

In [61]:
# Redução de Multicolinearidade no IDHM
# Mantemos apenas os indicadores macro para evitar redundância de informações
cols_idhm_manter = [
    'year', 'year_join', 'state_code', 'hdi', 'hdi_life_expectancy', 
    'hdi_education', 'hdi_income', 'gini_index', 'income_per_capita_mean', 'age_lag_2y_basic_rate'
]
    
idhm = idhm[cols_idhm_manter]

# Remoção de Redundâncias
# O FUNDEB mantém state_abbreviation e city_name_norm para o merge composto futuro.
fundeb = fundeb.drop(columns=['city_name'], errors='ignore')

# Bases com city_code/state_code intactos não precisam manter as colunas de texto originais
censo_2022 = censo_2022.drop(columns=['city_name'], errors='ignore')
idhm = idhm.drop(columns=['state_name'], errors='ignore')

# Validação das alterações
print(f"Colunas atuais do IDHM: {idhm.columns.tolist()}")
print(f"Colunas prontas no FUNDEB: {fundeb.columns.tolist()}")

Colunas atuais do IDHM: ['year', 'year_join', 'state_code', 'hdi', 'hdi_life_expectancy', 'hdi_education', 'hdi_income', 'gini_index', 'income_per_capita_mean', 'age_lag_2y_basic_rate']
Colunas prontas no FUNDEB: ['year', 'state_abbreviation', 'fundeb_amount', 'city_name_norm']


Agora iremos de fato realizar o merge propriamente dito. O dataset resultante será utilizado na fase de EDA. Optei por colocar os merges separados para que, caso ocorra algum erro, nós consigamos identificar de forma mais simples.

In [62]:
# Inicializa o dataset resultante e já limpa a base original da memória
df_EDA = literate_exam.copy()
del literate_exam
gc.collect()

# Merge Censo 2022
colunas_censo = [col for col in censo_2022.columns if col != 'state_abbreviation']
df_EDA = pd.merge(
    df_EDA, 
    censo_2022[colunas_censo], 
    on='city_code', 
    how='left'
)
del censo_2022
gc.collect()


# Validação final da base consolidada
print(f"Dimensões finais do dataset (df_EDA):")
print(f"Total de Linhas: {df_EDA.shape[0]}")
print(f"Total de Colunas: {df_EDA.shape[1]}")

Dimensões finais do dataset (df_EDA):
Total de Linhas: 6090138
Total de Colunas: 31


In [63]:
# Merge FUNDEB
df_EDA = pd.merge(
    df_EDA, 
    fundeb, 
    left_on=['city_name_norm', 'state_abbreviation', 'year'], 
    right_on=['city_name_norm', 'state_abbreviation', 'year'], 
    how='left',
    suffixes=('', '_fundeb')
)
del fundeb
gc.collect()


# Validação final da base consolidada
print(f"Dimensões finais do dataset (df_EDA):")
print(f"Total de Linhas: {df_EDA.shape[0]}")
print(f"Total de Colunas: {df_EDA.shape[1]}")

Dimensões finais do dataset (df_EDA):
Total de Linhas: 6090138
Total de Colunas: 32


In [64]:
# Merge Censo Escolar
df_EDA = pd.merge(
    df_EDA, 
    censo_escolar, 
    left_on=['city_code', 'year'], 
    right_on=['city_code', 'year'], 
    how='left', 
    suffixes=('', '_censo_escolar')
)
del censo_escolar
gc.collect()


# Validação final da base consolidada
print(f"Dimensões finais do dataset (df_EDA):")
print(f"Total de Linhas: {df_EDA.shape[0]}")
print(f"Total de Colunas: {df_EDA.shape[1]}")

Dimensões finais do dataset (df_EDA):
Total de Linhas: 6090138
Total de Colunas: 46


In [65]:
# Merge IDHM
df_EDA = pd.merge(
    df_EDA, 
    idhm, 
    left_on=['state_code', 'year'], 
    right_on=['state_code', 'year_join'], 
    how='left',
    suffixes=('', '_idhm')
)
del idhm
gc.collect()

# Validação final da base consolidada
print(f"Dimensões finais do dataset (df_EDA):")
print(f"Total de Linhas: {df_EDA.shape[0]}")
print(f"Total de Colunas: {df_EDA.shape[1]}")

Dimensões finais do dataset (df_EDA):
Total de Linhas: 6090138
Total de Colunas: 55


Agora iremos verificar se foram gerados nulos. Com exceção dos dados dos alunos qu estavam por conta das faltas, os outros datasets não possuíam valores nulos antes do merge.

In [66]:
df_EDA.isnull().sum()

year                                       0
state_code                                 0
student_id                                 0
grade                                      0
school_id                                  0
education_network_type                     0
city_code                                  0
student_is_present_exam                    0
student_took_exam                          0
student_exam_weight                   769406
student_exam_score                    769406
student_is_literate                        0
city_name                                  0
city_percentage_literate_students          0
city_average_exam_score                    0
state_abbreviation                         0
state_percentage_literate_students         0
state_average_exam_score                   0
city_name_norm                             0
households                               118
area                                     118
literacy_rate                            118
median_age

Encontramos alguns valores nulos após o merge. Provavelmente a ausência de dados foi gerada por ids de municípios ou dados de algum ano específico faltantes nos datasets de origem entre os dados do censo, do fundeb e de alfabetização. Iremos tratar esss dados faltantes de formas diferentes.

In [67]:
# 1. Filtro de alunos faltantes (remove os 769.406 nulos das notas e pesos)
df_EDA = df_EDA[df_EDA['student_is_present_exam'] == 1].copy()

# 2. Descarte dos nulos residuais do Censo 2022 (118 linhas)
df_EDA = df_EDA.dropna(subset=['population_total'])

# 3. Imputação dos nulos do FUNDEB (8.428 linhas) pela mediana do Estado/Ano
fundeb_imputer = df_EDA.groupby(['state_abbreviation', 'year'])['fundeb_amount'].transform('median')
df_EDA['fundeb_amount'] = df_EDA['fundeb_amount'].fillna(fundeb_imputer)

# Como não imputamos os faltantes, podemos dropar as colunas de presença/preenchimento 
# que agora terão variância zero (todos serão = 1)
df_EDA = df_EDA.drop(columns=['student_is_present_exam', 'student_took_exam'])

# Auditoria final do pipeline de limpeza
nulos_restantes = df_EDA.isnull().sum().sum()
print(f"Total de valores nulos no dataset final: {nulos_restantes}")
print(f"Dimensões prontas para modelagem: {df_EDA.shape}")

Total de valores nulos no dataset final: 0
Dimensões prontas para modelagem: (5320622, 53)


Evidentemente, não iremos alimentar o modelo com dados faltantes. Porém, estamos carregando os dados de presença por enquanto porque pensamos em criar variáveis no nível escola utilizando os dados porcentagem de presença por escola. 